# m26 lab · Map quality and uncertainty

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

A map should say where it is reliable. You interpolate activation with a Gaussian process and check that its uncertainty is honest, score a map by leave-one-out error without ground truth, add up the errors each point carries, relate point density to distance from the nearest point, and choose where to map next.

| Part | What you do | Equation |
| --- | --- | --- |
| A | A probabilistic activation map | eq-m26-gp |
| B | Leave-one-out without refitting | eq-m26-loo |
| C | The per-point error budget | eq-m26-budget |
| D | Point density | eq-m26-density |
| E | Where to map next | eq-m26-adaptive |

Times in ms, lengths in mm. Conduction velocity 1.0 m/s as in m03; the 1.0–1.5 points/cm² range is from Williams and colleagues; kernel settings and error sizes are illustrative. All data are synthetic. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import uncertainty as UN, interpolation as IP

## Part A · A probabilistic activation map

A Gaussian process with covariance $k$ gives, at any site, a mean $\mu = m + \mathbf k_*^\top(K + \sigma_n^2I)^{-1}(\mathbf y - m)$
and a variance $k_{**} - \mathbf k_*^\top(K + \sigma_n^2I)^{-1}\mathbf k_*$ (eq-m26-gp). Interpolate a focal activation from
40 noisy points and check how often the truth lies within $\pm1.96$ SD. Coveney and colleagues did this on atrial
surfaces.

In [ ]:
rng = np.random.default_rng(26)
src, c = np.array([12.0, 15.0]), 1.0
G = np.stack(np.meshgrid(np.linspace(0, 40, 41), np.linspace(0, 40, 41)), -1).reshape(-1, 2)
truth = IP.focal_time(G[:, 0], G[:, 1], c, *src)
X = rng.uniform(0, 40, (40, 2))
sn = 1.5                                                  # per-point error SD (ms), illustrative
y = IP.focal_time(X[:, 0], X[:, 1], c, *src) + rng.normal(0, sn, 40)
for ell in (4.0, 8.0, 16.0):
    mu, var = UN.gp_posterior(X, y, G, ell, 20.0**2, sn, y.mean())
    z = np.abs(truth - mu) / np.sqrt(var)
    print(f"length scale {ell:4.1f} mm: RMS error {np.sqrt(np.mean((mu - truth) ** 2)):.2f} ms, truth within 1.96 SD at {100 * np.mean(z <= 1.96):.0f}% of sites, largest SD {np.sqrt(var.max()):.1f} ms")
mu, var = UN.gp_posterior(X, y, G, 8.0, 20.0**2, sn, y.mean())
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a_, f, t in ((ax[0], mu, "mean (ms)"), (ax[1], np.sqrt(var), "SD (ms)")):
    im = a_.tricontourf(G[:, 0], G[:, 1], f, 20); a_.plot(X[:, 0], X[:, 1], "k.", ms=4); a_.set_title(t); a_.set_aspect("equal"); fig.colorbar(im, ax=a_)
plt.show()

In [ ]:
# Checks
assert np.sqrt(var).min() < sn and np.sqrt(var).max() > 3 * sn

## Part B · Leave-one-out without refitting

Without ground truth, judge a map by predicting each point from the others. For a GP this needs no refitting:
$r_i = [A^{-1}(\mathbf y - m)]_i/[A^{-1}]_{ii}$ with $A = K + \sigma_n^2I$ (eq-m26-loo; Rasmussen and Williams). Use it to choose
the length scale.

In [ ]:
for ell in (2.0, 4.0, 8.0, 16.0, 32.0):
    r, v = UN.loo_residuals(X, y, ell, 20.0**2, sn, y.mean())
    mu, _ = UN.gp_posterior(X, y, G, ell, 20.0**2, sn, y.mean())
    print(f"length scale {ell:4.1f} mm: LOO RMS {np.sqrt(np.mean(r**2)):.2f} ms, true map RMS error {np.sqrt(np.mean((mu - truth) ** 2)):.2f} ms")
r1, _ = UN.loo_residuals(X, y, 8.0, 400.0, sn, y.mean())
r2, _ = UN.loo_bruteforce(X, y, 8.0, 400.0, sn, y.mean())
print("closed form vs refitting, largest difference:", float(np.abs(r1 - r2).max()))

In [ ]:
# Checks
assert np.allclose(r1, r2, atol=1e-8)

## Part C · The per-point error budget

Each point carries annotation error $\sigma_a$ (m22), reference error $\sigma_r$ if points come from different beats
(m22), and position error $\sigma_x$ (m16–m19), which moves its time by $\sigma_x/c$ (eq-m26-budget). Independent errors
add in variance; the total is the GP's observation noise. Find which term dominates.

In [ ]:
for sa, sr, sx in ((1.0, 2.0, 2.0), (0.5, 0.0, 2.0), (2.0, 0.0, 0.5), (1.0, 3.0, 1.0)):
    tot = UN.error_budget(sa, sr, sx, c)
    parts = {"annotation": sa**2, "reference": sr**2, "position": (sx / c) ** 2}
    print(f"annotation {sa} ms, reference {sr} ms, position {sx} mm -> per-point SD {tot:.2f} ms, largest share: {max(parts, key=parts.get)}")
n = 200_000
e = rng.normal(0, 1.0, n) + rng.normal(0, 2.0, n) + rng.normal(0, 2.0, n) / c
print("simulated SD", round(float(np.std(e)), 3), "vs", round(UN.error_budget(1.0, 2.0, 2.0, c), 3))

In [ ]:
# Checks
assert abs(np.std(e) - UN.error_budget(1.0, 2.0, 2.0, c)) < 0.02

## Part D · Point density

For points placed at random with density $\rho$, the distance from a random site to the nearest point has mean
$1/(2\sqrt\rho)$ and median $\sqrt{\ln 2/(\pi\rho)}$ (eq-m26-density). At the 1.0–1.5 points/cm² that Williams and
colleagues found sufficient for atrial tachycardia maps, the mean is about 4–5 mm.

In [ ]:
for per_cm2 in (0.5, 1.0, 1.5, 3.0):
    rho = per_cm2 / 100.0
    P = rng.uniform(0, 100, (rng.poisson(rho * 1e4), 2))
    q = rng.uniform(25, 75, (2000, 2))
    d = np.min(np.linalg.norm(q[:, None, :] - P[None, :, :], axis=2), axis=1)
    print(f"{per_cm2:3.1f} points/cm^2: mean nearest distance {UN.nn_distance_mean(rho):.1f} mm (simulated {d.mean():.1f}), median {UN.nn_distance_median(rho):.1f} mm")

In [ ]:
# Checks
assert abs(UN.nn_distance_mean(0.01) - 5.0) < 1e-12

## Part E · Where to map next

The GP variance depends only on where the points are, not on what they measured, so a plan can be made before
mapping: put the next point where the variance is largest (eq-m26-adaptive). Compare 40 greedy points with 40 random
ones.

In [ ]:
cand = G
for name, idx in (("greedy", UN.greedy_max_variance(cand, 40, 8.0, 400.0, sn, start=int(np.argmin(np.linalg.norm(cand - 20.0, axis=1))))),
                  ("random", rng.choice(len(cand), 40, replace=False))):
    yy = truth[idx] + rng.normal(0, sn, 40)
    mu, var = UN.gp_posterior(cand[idx], yy, G, 8.0, 400.0, sn, yy.mean())
    print(f"{name:6s}: largest SD {np.sqrt(var.max()):.2f} ms, RMS error {np.sqrt(np.mean((mu - truth) ** 2)):.2f} ms")

In [ ]:
# Checks
g_idx = UN.greedy_max_variance(cand, 40, 8.0, 400.0, sn)
assert UN.gp_posterior(cand[g_idx], np.zeros(40), G, 8.0, 400.0, sn)[1].max() < UN.gp_posterior(cand[rng.choice(len(cand), 40, replace=False)], np.zeros(40), G, 8.0, 400.0, sn)[1].max()

## Where to go next

- Replace the Euclidean kernel with one on the mesh (m06 geodesics) so that uncertainty does not leak across a thin
  wall or a valve.
- Propagate the GP into conduction velocity (m25) by sampling maps from the posterior.
- m30 builds a full mapping pipeline; m46 discusses calibrated uncertainty for AI models.